# provework — GPU proving benchmark**What this does:** measures whether SP1's CUDA prover can prove our jobs on this GPU, and how fast. Two workloads:1. **V2 (SP1-native) job** — 1.79M zkVM cycles, the format new jobs use.2. **Legacy job through the emulator-in-a-guest** — 168.9M zkVM cycles, the format we measured as CPU-unprovable (OOM at 32 GB). This is the cell the whole experiment exists for.**Requirements:** a GPU runtime with ≥ 24 GB VRAM (L4 or A100 on Colab Pro). SP1's prover hard-refuses anything smaller — that refusal is itself a valid result, so just run it anyway.**Your part:** Runtime → Change runtime type → GPU (L4 or A100) → Run all → copy the summary at the bottom back to me.

In [ ]:
!nvidia-smiimport subprocess, reout = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()print('GPU:', out)vram = int(re.search(r'(\d+)\s*MiB', out).group(1)) if 'MiB' in out else 0print('VRAM OK (>= 24 GB):', vram >= 24000, f'({vram} MiB)')

In [ ]:
%%bash# System deps + Rust toolchain (~2 min)apt-get update -qq && apt-get install -y -qq git protobuf-compiler curl build-essential > /dev/nullcurl -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.98.1 --profile minimal > /dev/null 2>&1rustup target add riscv64imac-unknown-none-elf > /dev/null 2>&1echo "rust: $(rustc --version)"

In [ ]:
%%bash# Clone the repo and build the GPU prover/judge binary (~5-8 min, one-time)cd /content[ -d provework ] || git clone -q --depth 1 https://github.com/HolyWill90/proveworkcd proveworksource ~/.cargo/envcargo build --release -p sp1-host --features cuda 2>&1 | tail -2ls -la target/release/zk-judge

In [ ]:
%%bash# SP1's gpu-server needs the CUDA runtime library (the driver only provides libcuda)cd /contentwget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-cudart-12-6_12.6.77-1_amd64.debdpkg -x cuda-cudart-12-6_12.6.77-1_amd64.deb /content/cudaecho /content/cuda/usr/local/cuda-12.6/targets/x86_64-linux/lib > /etc/ld.so.conf.d/cudart.confldconfigldd ~/.sp1/bin/sp1-gpu-server 2>/dev/null | grep -c 'not found' || trueecho 'cudart installed'

In [ ]:
%%bash# BENCHMARK 1: V2 native job (~1.8M cycles) — the format new jobs usecd /content/proveworkmkdir -p /tmp/v2j && cp jobs/demo-v2/job.json jobs/demo-v2/input.bin jobs/demo-v2/program.elf /tmp/v2j/source ~/.cargo/env( nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv -l 5 > /tmp/vram1.log 2>&1 & )echo "monitor: $!"SP1_PROVER=cuda timeout 1800 target/release/zk-judge /tmp/v2j 10000000 elf/sp1-guest-emu /tmp/v2-receipt.binpkill -f 'nvidia-smi --query' 2>/dev/nullecho '--- peak VRAM:'; sort -t, -k1 -rn /tmp/vram1.log | head -2

In [ ]:
%%bash# BENCHMARK 2: legacy job through the emulator-in-a-guest (~169M cycles)# This is the job that OOM-killed a 32 GB CPU host at 31.8 GB.cd /content/proveworkmkdir -p /tmp/legj && cp jobs/demo-hash-smoke/job.json jobs/demo-hash-smoke/input.bin jobs/demo-hash-smoke/program.elf /tmp/legj/source ~/.cargo/env( nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv -l 5 > /tmp/vram2.log 2>&1 & )SP1_PROVER=cuda timeout 3600 target/release/zk-judge /tmp/legj 10000000000 elf/sp1-guest-emu /tmp/legacy-receipt.binRC=$?pkill -f 'nvidia-smi --query' 2>/dev/nullecho "exit=$RC (0 = proven, 124 = timed out at 60 min)"echo '--- peak VRAM:'; sort -t, -k1 -rn /tmp/vram2.log | head -2

In [ ]:
# SUMMARY — copy everything above (especially these lines) backimport subprocess, osprint('=' * 60)print('GPU BENCHMARK SUMMARY')print('=' * 60)print(open('/tmp/vram1.log').read().splitlines()[-1] if os.path.exists('/tmp/vram1.log') else 'v2: no data')print(open('/tmp/vram2.log').read().splitlines()[-1] if os.path.exists('/tmp/vram2.log') else 'legacy: no data')print('receipt files:', [f for f in ['/tmp/v2-receipt.bin', '/tmp/legacy-receipt.bin'] if os.path.exists(f)])print('Paste the full notebook output back to the project.')